In [ ]:
!pip install -q transformers accelerate timm pillow

In [ ]:
import torch
from transformers import AutoModelForImageSegmentation
from torchvision import transforms
from PIL import Image

In [ ]:
model = AutoModelForImageSegmentation.from_pretrained(
    "ZhengPeng7/BiRefNet",
    trust_remote_code=True
)

In [ ]:
uploaded = files.upload()

filename = list(uploaded.keys())[0]

image = cv2.imread(filename)
image = cv2.resize(image, (800, 800))

plt.figure(figsize=(8,8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Original Image")
plt.show()

In [ ]:
# ==========================================
# BACKGROUND REMOVAL
# ==========================================

# Convert to RGB
rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Remove background
result = remove(rgb)

# Convert RGBA to RGB if alpha channel exists
if result.shape[2] == 4:
    alpha = result[:, :, 3]
else:
    alpha = np.ones(result.shape[:2], dtype=np.uint8) * 255

# Binary meat mask
meat_mask = np.where(alpha > 127, 255, 0).astype(np.uint8)

# Clean mask
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7,7)
)

meat_mask = cv2.morphologyEx(
    meat_mask,
    cv2.MORPH_CLOSE,
    kernel
)

meat_mask = cv2.morphologyEx(
    meat_mask,
    cv2.MORPH_OPEN,
    kernel
)

plt.figure(figsize=(8,8))
plt.imshow(meat_mask, cmap='gray')
plt.title("AI Meat Mask")
plt.axis("off")
plt.show()

In [ ]:
meat = cv2.bitwise_and(image, image, mask=meat_mask)

plt.figure(figsize=(8,8))
plt.imshow(cv2.cvtColor(meat, cv2.COLOR_BGR2RGB))
plt.title("Segmented Meat")
plt.axis("off")
plt.show()

In [ ]:
# ==========================================
# HSV FAT DETECTION
# ==========================================

hsv = cv2.cvtColor(meat, cv2.COLOR_BGR2HSV)

lower_fat = np.array([0, 0, 120])
upper_fat = np.array([180, 120, 255])

fat_mask = cv2.inRange(hsv, lower_fat, upper_fat)

# Remove background
fat_mask = cv2.bitwise_and(fat_mask, meat_mask)

plt.figure(figsize=(8,8))
plt.imshow(fat_mask, cmap="gray")
plt.title("HSV Fat Mask")
plt.axis("off")
plt.show()

In [ ]:
fat_pixels = cv2.countNonZero(fat_mask)
meat_pixels = cv2.countNonZero(meat_mask)

fat_ratio = (fat_pixels / meat_pixels) * 100

print("="*40)
print(f"Fat Pixels : {fat_pixels:,}")
print(f"Meat Pixels: {meat_pixels:,}")
print(f"Fat Ratio : {fat_ratio:.2f}%")
print("="*40)